# IonQ — transpile only (no execution)

Mirror of `get_more_data_ibmq.ipynb`, but for IonQ. We **do not submit any jobs**.
For each circuit × IonQ noise-model target × optimization level, we:

1. Get the IonQ cloud-simulator backend with the noise-model that matches a real
   QPU (`aria-1`, `aria-2`, `forte-1`, `forte-enterprise-1`, `harmony`, `ideal`).
   The backend's `target` / native basis (`gpi`, `gpi2`, `ms` / `zz`) is what
   we transpile against.
2. Run Qiskit's preset pass manager → ISA circuit.
3. Capture pre-transpile (`algo_*`) and post-transpile (`hw_circ_*`) metrics
   exactly the way `QBacMet` Layer-1 / Layer-2 / Layer-3 do, then write a
   QBacMet-format JSON snapshot into `QBacMet/stats_sofar/` so
   `MQBac/0_preprocess.ipynb → flatten_qbacmet_json` picks them up automatically.
4. Append a flat row to `final_data/<USER>_ionq_data.csv`. `runtime_ms` is left
   `NaN` (no execution), `device="CPU"` so the runtime cascade in
   `0_preprocess.ipynb` correctly drops these from `best_backend_df` /
   `estimate_runtime_df` — they are *transpile metrics only*.

Edit `ionq_env.yaml` first (copy from `ionq_env.yaml.sample`).


In [ ]:
import json, os, sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml

from qiskit import QuantumCircuit
from qiskit.transpiler import preset_passmanagers

# qiskit-ionq is required (already installed in mqbacVirtEnv per env)
from qiskit_ionq import IonQProvider

# Make MQBac/bench_meta_gen.py importable for circuit constructors
BASE = Path.cwd()
sys.path.insert(0, str(BASE))
from bench_meta_gen import supermarq_metadata as _smq, tfim_metadata as _tfim  # noqa: F401

CFG_PATH = BASE / "ionq_env.yaml"
if not CFG_PATH.exists():
    raise FileNotFoundError(
        "Missing ionq_env.yaml in this folder. Copy ionq_env.yaml.sample and fill values."
    )

cfg = (yaml.safe_load(CFG_PATH.read_text()) or {}).get("CONFIG", {}) or {}
IONQ_API_KEY = (cfg.get("IONQ_API_KEY") or os.environ.get("IONQ_API_KEY") or "").strip()
IONQ_API_URL = (cfg.get("IONQ_API_URL") or os.environ.get("IONQ_API_URL") or "").strip()
USER_TAG = (cfg.get("USER_TAG") or os.environ.get("USER") or "user").strip().replace(" ", "_")
OUTPUT_DIR = Path(cfg.get("OUTPUT_DIR") or "./final_data"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR = (BASE / cfg.get("STATS_DIR", "../QBacMet/stats_sofar")).resolve()
STATS_DIR.mkdir(parents=True, exist_ok=True)

NOISE_MODELS = list(cfg.get("NOISE_MODELS") or ["ideal", "aria-1", "forte-1"])
OPT_LEVELS   = [int(x) for x in (cfg.get("OPT_LEVELS") or [0, 1, 2, 3])]
BENCHMARKS   = [tuple(x) for x in (cfg.get("BENCHMARKS") or [["ghz", 8], ["tfim", 8]])]

for px in ("HTTP_PROXY", "HTTPS_PROXY"):
    val = (cfg.get(px) or "").strip()
    if val:
        os.environ[px] = val

if not IONQ_API_KEY:
    raise ValueError("IONQ_API_KEY is required in ionq_env.yaml.")

provider = IonQProvider(IONQ_API_KEY, url=IONQ_API_URL or None)
print("IonQ provider ready.")
print("USER_TAG       =", USER_TAG)
print("OUTPUT_DIR     =", OUTPUT_DIR.resolve())
print("STATS_DIR      =", STATS_DIR)
print("NOISE_MODELS   =", NOISE_MODELS)
print("OPT_LEVELS     =", OPT_LEVELS)
print("BENCHMARKS     =", len(BENCHMARKS), "entries")


In [ ]:
# ── Circuit constructors (mirror what supermarq/TFIM benchmarks build) ──
def build_circuit(benchmark: str, n_qubits: int) -> QuantumCircuit:
    if benchmark == "tfim":
        from bench_meta_gen import QuantumCircuit as _QC  # qiskit's QC
        # Inline TFIM matching applications/TFIM constructor
        J, B, dt = 1.0, 0.5, 0.01
        qc = QuantumCircuit(n_qubits, n_qubits)
        for q in range(n_qubits):
            qc.rx(-2 * dt * B, q)
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
            qc.rz(-2 * dt * J, q + 1)
            qc.cx(q, q + 1)
        if n_qubits > 2:
            qc.cx(n_qubits - 1, 0)
            qc.rz(-2 * dt * J, 0)
            qc.cx(n_qubits - 1, 0)
        qc.measure(range(n_qubits), range(n_qubits))
        return qc

    if benchmark == "hhl":
        # qasm files live under applications/hhl/
        qasm_dir = (BASE.parent / "applications" / "hhl").resolve()
        candidates = sorted(qasm_dir.glob("*.qasm"))
        if not candidates:
            raise FileNotFoundError(f"No HHL qasm files at {qasm_dir}")
        # Pick first that matches n_qubits if specified; else just first
        for fp in candidates:
            try:
                qc = QuantumCircuit.from_qasm_file(str(fp))
                if qc.num_qubits == n_qubits:
                    return qc
            except Exception:
                continue
        return QuantumCircuit.from_qasm_file(str(candidates[0]))

    # Supermarq family
    import supermarq as sm
    if benchmark == "ghz":
        return sm.benchmarks.ghz.GHZ(num_qubits=n_qubits).qiskit_circuit()
    if benchmark == "ham":
        return sm.benchmarks.hamiltonian_simulation.HamiltonianSimulation(num_qubits=n_qubits).qiskit_circuit()
    if benchmark == "mermin_bell":
        return sm.benchmarks.mermin_bell.MerminBell(num_qubits=n_qubits).qiskit_circuit()
    if benchmark == "bit_code":
        return sm.benchmarks.bit_code.BitCode(
            num_data_qubits=n_qubits, num_rounds=1, bit_state=[0] * n_qubits
        ).qiskit_circuit()
    if benchmark == "phase_code":
        return sm.benchmarks.phase_code.PhaseCode(
            num_data_qubits=n_qubits, num_rounds=1, phase_state=[0] * n_qubits
        ).qiskit_circuit()

    raise ValueError(f"Unknown benchmark: {benchmark}")


# ── Metric extractors that match QBacMet Layer-1 / Layer-2 schema ──
_CLIFFORD_OPS = {
    "id", "x", "y", "z", "h", "s", "sdg", "sx", "sxdg",
    "cx", "cy", "cz", "swap", "measure", "barrier", "reset",
}


def _connectivity_stats(qc: QuantumCircuit):
    used, edges = set(), set()
    for inst, qargs, _ in qc.data:
        idxs = [qc.find_bit(q).index for q in qargs]
        used.update(idxs)
        if len(idxs) >= 2:
            for i in range(len(idxs)):
                for j in range(i + 1, len(idxs)):
                    a, b = sorted((idxs[i], idxs[j]))
                    edges.add((a, b))
    if not used:
        return 0, 0, 0.0
    parent = {u: u for u in used}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in edges:
        if a in parent and b in parent:
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    cc = len({find(u) for u in used})
    deg = {u: 0 for u in used}
    for a, b in edges:
        deg[a] += 1; deg[b] += 1
    deg_vals = list(deg.values()) or [0]
    return cc, max(deg_vals), float(sum(deg_vals)) / len(deg_vals)


def algo_metrics(qc: QuantumCircuit) -> dict:
    """Layer-1 metrics — pre-transpile (logical circuit)."""
    counts = qc.count_ops()
    one_q = two_q = three_q = 0
    op_names = []
    for inst, qargs, _ in qc.data:
        op_names.append(str(inst.name).lower())
        if len(qargs) == 1:
            one_q += 1
        elif len(qargs) == 2:
            two_q += 1
        elif len(qargs) >= 3:
            three_q += 1
    cc, _, _ = _connectivity_stats(qc)
    n_cliff = sum(1 for n in op_names if n in _CLIFFORD_OPS)
    n_non = len(op_names) - n_cliff
    return {
        "num_qubits": int(qc.num_qubits),
        "num_clbits": int(qc.num_clbits),
        "depth": int(qc.depth()),
        "width": int(qc.width()),
        "num_ops": int(qc.size()),
        "num_2q_gates": two_q,
        "num_cliffords": int(n_cliff),
        "num_non_cliffords": int(n_non),
        "critical_path_length": int(qc.depth()),
        "connected_components": int(cc),
        "num_parameters": int(len(qc.parameters)),
        "gate_counts": {str(k): int(v) for k, v in counts.items()},
        "framework": "qiskit",
    }


def transpile_metrics(hw_qc: QuantumCircuit) -> dict:
    """Layer-2 metrics — post-transpile (ISA circuit)."""
    counts = dict(hw_qc.count_ops()) if hw_qc.count_ops() else {}
    cc, max_deg, avg_deg = _connectivity_stats(hw_qc)
    return {
        "hw_circ_width": int(hw_qc.num_qubits),
        "hw_circ_depth": int(hw_qc.depth()),
        "hw_circ_num_ops": int(hw_qc.size()),
        "hw_circ_gate_counts": {str(k): int(v) for k, v in counts.items()},
        "hw_circ_swaps": int(counts.get("swap", 0)),
        "hw_circ_num_2q_gates": int(sum(counts.get(g, 0) for g in ["cx", "cz", "swap", "ms", "zz"])),
        "hw_circ_connected_components": int(cc),
        "hw_circ_connectivity_degree_max": int(max_deg),
        "hw_circ_connectivity_degree_avg": float(avg_deg),
    }


def backend_features(backend) -> dict:
    """Layer-3 — match QBacMet BackendLayer.set_backend_details exactly."""
    name = getattr(backend, "name", None)
    if callable(name):
        try: name = name()
        except Exception: name = None
    version = getattr(backend, "version", None)
    if callable(version):
        try: version = version()
        except Exception: version = None
    try:
        mod = backend.__class__.__module__
    except Exception:
        mod = None
    name_l = (str(name).lower() if name else "")
    is_sim = ("aer" in (mod or "").lower()) or ("simulator" in name_l)
    return {
        "backend_name": name,
        "backend_version": version,
        "backend_module": mod,
        "backend_is_simulator": bool(is_sim),
        "backend_type": "simulator" if is_sim else "hardware",
        "env_backend": os.environ.get("QBACMET_BACKEND"),
    }


In [ ]:
# ── Main loop: transpile every (benchmark, n_qubits) × noise_model × opt_level ──
import time, traceback

rows = []
written_jsons = 0
n_total = len(BENCHMARKS) * len(NOISE_MODELS) * len(OPT_LEVELS)
print(f"Planned transpile jobs: {n_total}\n")

# Cache built circuits and backend objects to avoid repeated work
_circ_cache  = {}
_be_cache    = {}

def _get_backend(noise_model: str):
    if noise_model in _be_cache:
        return _be_cache[noise_model]
    # IonQ cloud-simulator backend with a hardware-flavored noise model.
    # Even with noise_model="ideal" the backend's target reflects the
    # simulator's gate set, which is what we transpile against.
    be = provider.get_backend("simulator", noise_model=noise_model)
    _be_cache[noise_model] = be
    return be

def _get_circuit(benchmark: str, n_qubits: int):
    key = (benchmark, n_qubits)
    if key in _circ_cache:
        return _circ_cache[key]
    qc = build_circuit(benchmark, n_qubits)
    _circ_cache[key] = qc
    return qc

i = 0
for benchmark, nq in BENCHMARKS:
    nq = int(nq)
    try:
        qc = _get_circuit(benchmark, nq)
    except Exception as e:
        print(f"  [skip] {benchmark} nq={nq}: build failed: {e}")
        continue
    a_metrics = algo_metrics(qc)

    for nm in NOISE_MODELS:
        try:
            backend = _get_backend(nm)
        except Exception as e:
            print(f"  [skip] noise_model={nm}: backend init failed: {e}")
            continue
        be_feats = backend_features(backend)
        # Slight reshape: surface the noise model as the sub_backend label,
        # matching the existing IonQ snapshot convention (sub_backend='simulator').
        # We store the noise model in `args.sub_backend` instead so analysis can
        # distinguish aria-1 vs forte-1 transpile passes.
        for opt in OPT_LEVELS:
            i += 1
            t0 = time.perf_counter()
            try:
                pm = preset_passmanagers.generate_preset_pass_manager(
                    target=getattr(backend, "target", None),
                    optimization_level=opt,
                )
                hw_qc = pm.run(qc)
                t_transpile = time.perf_counter() - t0
                t_metrics = transpile_metrics(hw_qc)
            except Exception as e:
                print(f"  [err] {benchmark} nq={nq} nm={nm} opt={opt}: {e}")
                traceback.print_exc(limit=1)
                continue

            ts_iso = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

            # ── QBacMet-format JSON snapshot (matches stats_sofar/ schema) ──
            payload = {
                "args": {
                    "benchmark_name": benchmark,
                    "number_of_qubits": nq,
                    "simulator_type": "ionq",
                    "sub_backend": nm,         # noise-model name (aria-1, forte-1, ...)
                    "device": "CPU",            # transpile only — no execution
                    "run_mode": "transpile",   # disambiguates from sync runs
                    "number_of_iterations": 0,
                    "optimization_level": opt,
                },
                "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
                "layers": {
                    "layer_0_slurm":      {"features": {}, "metrics": {}},
                    "layer_1_algorithm":  {"features": {}, "metrics": a_metrics},
                    "layer_2_transpile":  {"features": {"optimization_level": opt,
                                                         "noise_model": nm},
                                            "metrics": t_metrics},
                    "layer_3_backend":    {"features": be_feats, "metrics": {}},
                    "layer_4_execution":  {"features": {}, "metrics": {
                        "transpile_time": t_transpile,
                    }},
                    "layer_5_postprocess":{"features": {}, "metrics": {}},
                },
                "raw_sources": {"frontend": {"note": "transpile-only via get_more_data_ionq.ipynb"}},
            }
            fname = (
                f"{benchmark}_nq{nq}_ionq_{nm}_CPU_transpile_itrs0_opt{opt}_{ts_iso}.json"
            )
            (STATS_DIR / fname).write_text(json.dumps(payload, indent=2, default=str))
            written_jsons += 1

            # ── Flat CSV row (mirrors get_more_data_ibmq.ipynb schema) ──
            rows.append({
                "provider":     "ionq",
                "backend":      nm,
                "sub_backend":  nm,
                "benchmark":    benchmark,
                "n_qubits":     nq,
                "device":       "CPU",          # transpile-only
                "run_mode":     "transpile",
                "n_nodes":      1,
                "n_processes":  1,
                "opt_level":    opt,
                "transpile_time_s": t_transpile,
                "runtime_ms":   None,            # never executed
                "depth":        a_metrics["depth"],
                "n_ops":        a_metrics["num_ops"],
                "single_qubit_gates": sum(1 for g, c in a_metrics["gate_counts"].items()
                                          if g not in ("measure", "barrier", "reset")
                                          and c and g not in ("cx", "cz", "swap", "ms")),
                "two_qubit_gates":    a_metrics["num_2q_gates"],
                "three_qubit_gates":  0,
                "measure_ops":        int(a_metrics["gate_counts"].get("measure", 0)),
                "num_cliffords":      a_metrics["num_cliffords"],
                "num_non_cliffords":  a_metrics["num_non_cliffords"],
                "num_parameters":     a_metrics["num_parameters"],
                "critical_path_length": a_metrics["critical_path_length"],
                "connected_components": a_metrics["connected_components"],
                "gate_counts_json":    json.dumps(a_metrics["gate_counts"]),
                "hw_circ_depth":       t_metrics["hw_circ_depth"],
                "hw_circ_num_ops":     t_metrics["hw_circ_num_ops"],
                "hw_circ_num_2q_gates":t_metrics["hw_circ_num_2q_gates"],
                "hw_circ_swaps":       t_metrics["hw_circ_swaps"],
                "hw_circ_gate_counts_json": json.dumps(t_metrics["hw_circ_gate_counts"]),
                "circuit_metrics_json": json.dumps({"algo": a_metrics, "hw": t_metrics}),
            })

            if i % 25 == 0 or i == n_total:
                print(f"  [{i}/{n_total}] {benchmark} nq={nq} nm={nm} opt={opt} "
                      f"depth {a_metrics['depth']} → hw {t_metrics['hw_circ_depth']} "
                      f"({t_transpile*1000:.1f} ms)")

print(f"\nWritten {written_jsons} QBacMet snapshots → {STATS_DIR}")
print(f"Collected {len(rows)} flat rows")


In [ ]:
# ── Save the flat CSV (mirror IBMQ output naming: <USER>_ionq_data.csv) ──
df = pd.DataFrame(rows)
out_path = OUTPUT_DIR / f"{USER_TAG}_ionq_data.csv"
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df)} rows)")

if not df.empty:
    print("\n── Per noise-model summary (transpile overhead) ──")
    g = df.groupby(["sub_backend", "opt_level"]).agg(
        n=("benchmark", "size"),
        algo_depth_mean=("depth", "mean"),
        hw_depth_mean=("hw_circ_depth", "mean"),
        algo_2q_mean=("two_qubit_gates", "mean"),
        hw_2q_mean=("hw_circ_num_2q_gates", "mean"),
        transpile_ms_mean=("transpile_time_s", lambda s: float(s.mean()) * 1000.0),
    ).round(2)
    print(g.to_string())

    print("\n── Per benchmark depth inflation (averaged across noise models, opt=1) ──")
    sub = df[df["opt_level"] == 1]
    if not sub.empty:
        g2 = (sub.groupby("benchmark")
                  .agg(algo_depth_mean=("depth", "mean"),
                       hw_depth_mean=("hw_circ_depth", "mean"))
                  .assign(inflation=lambda d: (d["hw_depth_mean"] / d["algo_depth_mean"]).round(2))
                  .round(2))
        print(g2.to_string())

print("\nDone. Re-run MQBac/0_preprocess.ipynb to incorporate the new IonQ "
      "transpile snapshots into qbacmet_flat.csv.")
